In [1]:
import pandas as pd
df =pd.read_csv("online_retail_cleaned.csv", header=None)
df.columns=['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price' , 'CustomerID', 'Country']
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'],errors='coerce')
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")

df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

print(df.shape)
df.info()


C:\Users\Rajesh Srivastav\AppData\Local\Temp\ipykernel_8240\1811320129.py:2: DtypeWarning: Columns (0,3,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df =pd.read_csv("online_retail_cleaned.csv", header=None)


(1033036, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033036 non-null  object        
 1   StockCode    1033036 non-null  object        
 2   Description  1028761 non-null  object        
 3   Quantity     1033025 non-null  float64       
 4   InvoiceDate  1033025 non-null  datetime64[ns]
 5   Price        1033034 non-null  float64       
 6   CustomerID   1033036 non-null  object        
 7   Country      1033025 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(5)
memory usage: 63.1+ MB


In [2]:
# filter and calculate recency,frequency,monetary 
rfm_data = df[
    (df["CustomerID"].notna()) &
    (df["Quantity"] > 0)
].copy()

snapshot_date = pd.to_datetime('2011-12-10')

rfm = rfm_data.groupby("CustomerID").agg(
    recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    frequency=('Invoice', 'nunique'),
    monetary=('Quantity', lambda x: (
        x * rfm_data.loc[x.index, 'Price']
    ).sum())
).reset_index()
rfm.head()

,CustomerID,recency,frequency,monetary
0,12346,325,12,77556.46
1,12347,2,8,4921.53
2,12348,75,5,2019.40
3,12349,18,4,4428.69
4,12350,310,1,334.40


In [3]:
rfm['R_Score'] = pd.qcut(
    rfm['recency'].rank(method='first'),
    4,
    labels=[4, 3, 2, 1]
)

rfm['F_Score'] = pd.qcut(
    rfm['frequency'].rank(method='first'),
    4,
    labels=[1, 2, 3, 4]
)

rfm['M_Score'] = pd.qcut(
    rfm['monetary'].rank(method='first'),
    4,
    labels=[1, 2, 3, 4]
)
rfm['RFM_Score'] = (
    rfm['R_Score'].astype(str) +
    rfm['F_Score'].astype(str) +
    rfm['M_Score'].astype(str)
)
def customer_segment(row):

    if row['R_Score'] >= 3 and row['F_Score'] >= 3 and row['M_Score'] >= 3:
        return 'Champions'

    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal Customers'

    elif row['R_Score'] >= 3 and row['F_Score'] <= 2:
        return 'Recent Customers'

    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'

    else:
        return 'Lost Customers'


rfm['Segment'] = rfm.apply(customer_segment, axis=1)
rfm['Ischurned'] =rfm['recency'] >90


rfm.head()

,CustomerID,recency,frequency,monetary,R_Score,F_Score,M_Score,RFM_Score,Segment,Ischurned
0,12346,325,12,77556.46,2,4,4,244,At Risk,True
1,12347,2,8,4921.53,4,4,4,444,Champions,False
2,12348,75,5,2019.40,3,3,3,333,Champions,False
3,12349,18,4,4428.69,4,3,4,434,Champions,False
4,12350,310,1,334.40,2,1,1,211,Lost Customers,True


In [4]:
# Merge rfm result is back into transactions and export
df=df[(df['CustomerID'].notna()) & (df['Quantity']>0)].copy()
df['InvoiceMonth']=df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
df['Revenue']=df['Quantity'] * df['Price']
df_final=df.merge(
    rfm[['CustomerID','recency','frequency','monetary','R_Score','F_Score','M_Score','Segment']],
    on='CustomerID', how='left'
)
df_final.to_csv('EcommerceDashboarddata.xlsx',index= False,encoding='utf-8-sig')